# 09 - Quick fix: test(30%) predictions + LLM input rows

This notebook creates demo artifacts from **saved ensemble outputs**:

1. Load saved base-model `test` probabilities and saved `weights.json`.
2. Sample 30% of the processed test split (deterministic seed).
3. Save prediction records with full per-model probabilities.
4. Build structured `prompt/completion` rows (same schema as notebook 05) for LLM inference.

Outputs are written to `tm_research/ensemble/artifacts/`.

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

REPO_ROOT = '/content/drive/MyDrive/thesis/topicmodeling/tm_research'
TMP_ROOT = '/content/ensemble_tmp'
REPO_PATH = Path(REPO_ROOT)
IMPORT_ROOT = str(REPO_PATH.parent if REPO_PATH.name == 'tm_research' else REPO_PATH)

if 'google.colab' in sys.modules:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
    if IMPORT_ROOT not in sys.path:
        sys.path.append(IMPORT_ROOT)
else:
    import pathlib
    LOCAL = pathlib.Path.cwd().resolve()
    while LOCAL.name != 'tm_research' and LOCAL.parent != LOCAL:
        LOCAL = LOCAL.parent
    repo_parent = str(LOCAL.parent)
    if repo_parent not in sys.path:
        sys.path.append(repo_parent)

import numpy as np
import pandas as pd

from tm_research.ensemble.colab_setup import setup_colab
paths = setup_colab(repo_root=REPO_ROOT, tmp_root=TMP_ROOT)

from tm_research.ensemble.utils_io import load_splits
from tm_research.ensemble.utils_stacking import BASE_MODEL_NAMES, format_prompt, weighted_average, write_jsonl

print('Repo root:', REPO_PATH)
print('Output dir:', REPO_PATH / 'ensemble' / 'artifacts')

In [ ]:
# Config
SAMPLE_FRACTION = 0.30
SAMPLE_SEED = 42
SPLIT_NAME = 'test'  # quick fix target split

# Derive save path directly from REPO_PATH (defined in cell 1) so it always
# resolves to the repo on Drive, regardless of env var or import order.
OUTPUT_DIR = REPO_PATH / 'ensemble' / 'artifacts'
META_JSONL_OUTPUT_DIR = OUTPUT_DIR / 'meta_jsonl'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
META_JSONL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PREDICTIONS_CSV = OUTPUT_DIR / f'demo_{SPLIT_NAME}_30_predictions_with_probs.csv'
LLM_INPUT_JSONL = META_JSONL_OUTPUT_DIR / f'demo_{SPLIT_NAME}_30_llm_input.jsonl'
LLM_INPUT_CSV = OUTPUT_DIR / f'demo_{SPLIT_NAME}_30_llm_input.csv'

print('Will save to repo:', OUTPUT_DIR)
print(' -', PREDICTIONS_CSV)
print(' -', LLM_INPUT_JSONL)
print(' -', LLM_INPUT_CSV)

In [ ]:
# Load data split + artifacts — all paths go directly to PERSISTENT_ARTIFACTS_DIR (repo on Drive)
train_df, val_df, test_df, label_map = load_splits()
split_df = {'train': train_df, 'val': val_df, 'test': test_df}[SPLIT_NAME].reset_index(drop=True)

weights_path = OUTPUT_DIR / 'weights.json'
if not weights_path.exists():
    raise FileNotFoundError(f'Missing required artifact: {weights_path}')
with open(weights_path, 'r', encoding='utf-8') as f:
    weights_payload = json.load(f)
weights = weights_payload.get('weights', weights_payload)

missing = [m for m in BASE_MODEL_NAMES if m not in weights]
if missing:
    raise ValueError(f'Missing weights for models: {missing}')

# Load .npy directly from the repo's probs/ dir
PROBS_DIR = OUTPUT_DIR / 'probs'
probs_per_model_full = {m: np.load(PROBS_DIR / f'{m}_{SPLIT_NAME}.npy') for m in BASE_MODEL_NAMES}
for m in BASE_MODEL_NAMES:
    if len(probs_per_model_full[m]) != len(split_df):
        raise ValueError(
            f'Length mismatch for {m}/{SPLIT_NAME}: probs={len(probs_per_model_full[m])}, rows={len(split_df)}'
        )

n_full = len(split_df)
k = max(1, int(round(n_full * SAMPLE_FRACTION)))
rs = np.random.RandomState(SAMPLE_SEED)
sample_idx = np.sort(rs.choice(n_full, size=k, replace=False))

sample_df = split_df.iloc[sample_idx].reset_index(drop=True)
probs_per_model = {m: probs_per_model_full[m][sample_idx] for m in BASE_MODEL_NAMES}

print(f'Classes: {label_map.class_names}')
print(f'Sampled {len(sample_df)}/{n_full} rows from {SPLIT_NAME} (fraction={SAMPLE_FRACTION}, seed={SAMPLE_SEED})')

In [ ]:
# Build prediction table with probability columns
weighted_probs = weighted_average(probs_per_model, weights)
weighted_idx = weighted_probs.argmax(axis=1)
weighted_conf = weighted_probs.max(axis=1)
weighted_labels = [label_map.id2label[int(i)] for i in weighted_idx]

result_df = pd.DataFrame({
    'orig_idx': sample_idx,
    'text': sample_df['text'].astype(str).tolist(),
    'gold_label': sample_df['label'].astype(str).tolist(),
    'pred_weighted': weighted_labels,
    'conf_weighted': weighted_conf.astype(float),
})

for m in BASE_MODEL_NAMES:
    arr = probs_per_model[m]
    for j, lab in enumerate(label_map.class_names):
        result_df[f'{m}__{lab}'] = arr[:, j].astype(float)

for j, lab in enumerate(label_map.class_names):
    result_df[f'weighted__{lab}'] = weighted_probs[:, j].astype(float)

PREDICTIONS_CSV.parent.mkdir(parents=True, exist_ok=True)
result_df.to_csv(PREDICTIONS_CSV, index=False, encoding='utf-8')

print('Saved prediction records with probabilities to:', PREDICTIONS_CSV)
print('Rows:', len(result_df), 'Columns:', len(result_df.columns))
print(result_df[['orig_idx', 'text', 'gold_label', 'pred_weighted', 'conf_weighted']].head(5).to_string(index=False))

In [ ]:
# Build structured LLM input rows (same prompt schema as notebook 05)
rows = []
for i in range(len(result_df)):
    per_model = {m: probs_per_model[m][i] for m in BASE_MODEL_NAMES}
    formatted = format_prompt(
        text=result_df.loc[i, 'text'],
        probs_per_model=per_model,
        weights=weights,
        label_map=label_map,
        label=None,  # test inference input: no gold label in completion
    )
    rows.append({
        'idx': int(result_df.loc[i, 'orig_idx']),
        'text': result_df.loc[i, 'text'],
        'prompt': formatted['prompt'],
        'completion': formatted['completion'],
        'gold': result_df.loc[i, 'gold_label'],
        'pred_weighted': result_df.loc[i, 'pred_weighted'],
        'conf_weighted': float(result_df.loc[i, 'conf_weighted']),
    })

write_jsonl(rows, LLM_INPUT_JSONL)

llm_csv_df = pd.DataFrame(rows)
llm_csv_df.to_csv(LLM_INPUT_CSV, index=False, encoding='utf-8')

print('Saved LLM structured input JSONL to:', LLM_INPUT_JSONL)
print('Saved LLM structured input CSV to:', LLM_INPUT_CSV)
print('Rows:', len(rows))
print('\nFirst prompt:\n')
print(rows[0]['prompt'])

In [ ]:
# Quick sanity checks
required_pred_cols = {'text', 'gold_label', 'pred_weighted', 'conf_weighted'}
missing_pred_cols = required_pred_cols - set(result_df.columns)
if missing_pred_cols:
    raise ValueError(f'Missing expected prediction columns: {missing_pred_cols}')

if len(rows) != len(result_df):
    raise ValueError(f'Row count mismatch: rows={len(rows)} result_df={len(result_df)}')

empty_prompts = sum(1 for r in rows if not str(r['prompt']).strip())
if empty_prompts:
    raise ValueError(f'Found {empty_prompts} empty prompts in LLM input rows')

print('Sanity checks passed.')
print('Prediction file exists:', Path(PREDICTIONS_CSV).exists())
print('JSONL file exists:', Path(LLM_INPUT_JSONL).exists())
print('CSV file exists:', Path(LLM_INPUT_CSV).exists())